# Dever de Casa — Regularização e Fine-Tuning Parcial em CIFAR-10
**ENG4502 — Introdução à Ciência de Dados · PUC-Rio**

Nesta atividade prática para casa, você aprofundará os experimentos de Transfer Learning desenvolvidos em aula. O objetivo é duplo:
1. **Avaliar a Regularização:** Investigar se a técnica de **Data Augmentation** ajuda a mitigar o overfitting do modelo treinado do zero (Scratch).
2. **Implementar Fine-Tuning Parcial (Eficiência):** Implementar o descongelamento seletivo de camadas. Vamos treinar apenas as últimas camadas convolucionais (`layer4`) e a camada final linear (`fc`) usando **Taxas de Aprendizado Discriminativas**.

## 🛠️ Instruções
- Preencha as seções marcadas com `### SEU CÓDIGO AQUI ###`.
- Utilize o mesmo subset balanceado do CIFAR-10 carregado em aula (já configurado no script).

## 0. Imports e Dispositivo

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Subset
import numpy as np
import matplotlib.pyplot as plt
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando o dispositivo: {device}')

## 1. Carregamento dos Dados com Data Augmentation

Para a parte de treino, além de redimensionar e normalizar, você deve aplicar transformações de regularização:
- Inversão horizontal aleatória (`RandomHorizontalFlip`).
- Rotação sutil aleatória (`RandomRotation`) de até 15 graus.
- Redimensionar para 224x224.

In [ ]:
# ==========================================
# EXERCÍCIO 1: Defina os pipelines de transformações
# Adicione Data Augmentation (Horizontal Flip e Rotation 15) em 'train'.
# Em 'val', aplique apenas o Resize, ToTensor e Normalize.
# Dica: Use o Normalize padrão do ImageNet: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]
# ==========================================

data_transforms = {
    'train': transforms.Compose([
        ### SEU CÓDIGO AQUI ###
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

# Carregamento completo
train_full = datasets.CIFAR10(root='data', train=True, download=True, transform=data_transforms['train'])
val_full = datasets.CIFAR10(root='data', train=False, download=True, transform=data_transforms['val'])

# Função para extrair subconjunto balanceado
def extract_balanced_subset(dataset, n_total):
    n_per_class = n_total // 10
    indices = []
    class_counts = {c: 0 for c in range(10)}
    for idx, (_, label) in enumerate(dataset):
        if class_counts[label] < n_per_class:
            indices.append(idx)
            class_counts[label] += 1
        if len(indices) == n_total:
            break
    return Subset(dataset, indices)

train_dataset = extract_balanced_subset(train_full, 5000)
val_dataset = extract_balanced_subset(val_full, 1000)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)

print(f'Treino: {len(train_dataset)} imagens | Validação: {len(val_dataset)} imagens')

## Parte A — Treinamento do Zero (Scratch) com Augmentation

Aqui, você instanciará um modelo sem pesos pré-treinados e medirá o comportamento de convergência após 5 épocas usando a sua base de treino com regularização.

In [ ]:
# ==========================================
# EXERCÍCIO 2: Carregue um modelo ResNet-18 sem pesos pré-treinados
# Altere model.fc para ter 10 saídas e envie para o device.
# ==========================================
### SEU CÓDIGO AQUI ###
model_scratch = None



### Loops de Treinamento e Validação

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(dataloader.dataset)

def evaluate(model, dataloader):
    model.eval()
    corrects = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            corrects += torch.sum(preds == labels.data)
    return corrects.double().item() / len(dataloader.dataset)

In [ ]:
# Otimizador e critério para o Scratch
criterion = nn.CrossEntropyLoss()
optimizer_scratch = optim.SGD(model_scratch.parameters(), lr=0.01, momentum=0.9)

# Treine por 5 épocas e armazene a acurácia de validação final
for epoch in range(1, 6):
    t0 = time.time()
    loss = train_epoch(model_scratch, train_loader, criterion, optimizer_scratch)
    acc = evaluate(model_scratch, val_loader)
    print(f'Época {epoch}/5 | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | Tempo: {time.time()-t0:.1f}s')

## Parte B — Fine-Tuning Parcial (layer4 + fc)

Neste exercício, você carregará o modelo com os pesos pré-treinados, mas fará o seguinte:
1. Congelará todos os parâmetros convolucionais do modelo inicialmente.
2. Descongelará apenas os parâmetros dentro do bloco `model.layer4`.
3. Substituirá a camada final `model.fc` por uma nova linear de 10 classes.
4. Criará um otimizador configurado para treinar ambos os blocos, utilizando taxas de aprendizado distintas (taxas discriminativas).

In [ ]:
# ==========================================
# EXERCÍCIO 3: Carregue o modelo ResNet-18 Pré-Treinado
# 1. Carregue com models.ResNet18_Weights.IMAGENET1K_V1
# 2. Congele todos os parâmetros
# 3. Descongele explicitamente model.layer4
# 4. Substitua model.fc
# ==========================================
### SEU CÓDIGO AQUI ###
model_partial = None


model_partial = model_partial.to(device)

# Validação do congelamento seletivo
total_params = sum(p.numel() for p in model_partial.parameters())
trainable_params = sum(p.numel() for p in model_partial.parameters() if p.requires_grad)
print(f'Total de Parâmetros: {total_params:,}')
print(f'Parâmetros Treináveis: {trainable_params:,} (deve ser aproximadamente 2.624.522)')

### Taxas de Aprendizado Discriminativas

Agora configure o otimizador SGD para passar um dicionário com dois grupos de parâmetros:
- Camadas convolucionais de `model.layer4` com `lr = 0.0001` (para refinar sem apagar o conhecimento).
- Camada final `model.fc` com `lr = 0.001` (aprendizado mais rápido).

In [ ]:
# ==========================================
# EXERCÍCIO 4: Crie o otimizador discriminativo
# ==========================================
### SEU CÓDIGO AQUI ###
optimizer_partial = None

# Treine por 5 épocas
for epoch in range(1, 6):
    t0 = time.time()
    loss = train_epoch(model_partial, train_loader, criterion, optimizer_partial)
    acc = evaluate(model_partial, val_loader)
    print(f'Época {epoch}/5 | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | Tempo: {time.time()-t0:.1f}s')

## Questões para Discussão

1. **Comparando o Scratch com e sem regularização:** O uso de Data Augmentation melhorou a acurácia de validação do modelo treinado do zero ao final da Época 5 em relação ao benchmark de aula (37.8%)? Explique o porquê.
   *Sua resposta aqui...*

2. **Tradeoff do Fine-Tuning Parcial:** Em relação à acurácia e ao tempo observados em aula, o Fine-Tuning Parcial conseguiu aproximar-se do Fine-Tuning Completo? Explique por que treinar apenas a `layer4` (2.6M de parâmetros) é mais eficiente e seguro em datasets menores.